In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tkinter import Tk, filedialog
import scipy.io
import os
%matplotlib qt5

In [2]:
rho = 38.38 / 32
speed_range = np.arange(0, 16, 2)
dynamic_pressure = 0.5 * rho * speed_range**2
dynamic_pressure

array([  0.     ,   2.39875,   9.595  ,  21.58875,  38.38   ,  59.96875,
        86.355  , 117.53875])

In [4]:
results_dir = '.\\results\\C'
files = [f for f in os.listdir(results_dir) if os.path.isfile(os.path.join(results_dir, f))]
print(files)

['U6_D16_55_RPM2000.mat', 'U6_D16_55_RPM2500.mat', 'U6_D16_55_RPM3000.mat', 'U6_D16_55_RPM3500.mat']


In [ ]:
file = "U6_D16_55_RPM2500.mat"
file_path = os.path.join(results_dir, file)

data = scipy.io.loadmat(file_path)

ai = data['ai'].flatten()
encoder_deg = data['encoder_deg'].flatten()
time = data['time'].flatten()
ai_rate_hz = data['ai_rate_hz'].flatten()
device = data['device'].flatten()
ai_channels = data['ai_channels'].flatten()
ppr = data['ppr'].flatten()
notes = data['notes'].flatten()
created_utc = data['created_utc'].flatten()
bias_pre = data['bias_pre']
bias_pre_time = data['bias_pre_time'].flatten()
bias_post = data['bias_post']
bias_post_time = data['bias_post_time'].flatten()
force_N = data['force_N']
force_time = data['force_time'].flatten()
manometers = data['manometers']
manometers_time = data['manometers_time'].flatten()
temperature = data['temperature'].flatten()
pressure = data['pressure'].flatten()
humidity = data['humidity'].flatten()
rho = data['rho'].flatten()
U_inf = data['U_inf'].flatten()
D_inch = data['D_inch'].flatten()
D_m = data['D_m'].flatten()
pitch = data['pitch'].flatten()
RPM = data['RPM'].flatten()
turbulence_case = data['turbulence_case'].flatten()
phase_angle = data['phase_angle'].flatten()

force_net = force_N.T - np.mean(bias_pre)
Fx, Fy, Fz, Mx, My, Mz = force_net[:, 0], force_net[:, 1], force_net[:, 2], force_net[:, 3], force_net[:, 4], force_net[:, 5]

time = time - time[0]

omega = RPM / 60 * 2 * np.pi  # rad/s
R_m = D_m[0] / 2
prop_area = np.pi * R_m**2
tip_speed = (D_m / 2) * omega  # m/s
mu = U_inf / tip_speed

mean_thrust = np.mean(-Fz)
mean_torque = np.mean(-Mz)
mean_power = mean_torque * omega

U_inf = U_inf[0]
RPM = RPM[0]

CT = mean_thrust / (rho * prop_area * tip_speed**2)
CQ = mean_torque / (rho * prop_area * tip_speed**2 * R_m)
CP = mean_power / (rho * prop_area * tip_speed**3)
print(CT, CQ, CP)

FileNotFoundError: [Errno 2] No such file or directory: '.\\results\\C\\U4_D16_55_RPM2500.mat'

In [19]:
mu_list = []
CT_list = []
CQ_list = []
CP_list = []
RPM_values = []

for file in files:
    file_path = os.path.join(results_dir, file)
    
    
    data = scipy.io.loadmat(file_path)
    
    ai = data['ai'].flatten()
    encoder_deg = data['encoder_deg'].flatten()
    time = data['time'].flatten()
    ai_rate_hz = data['ai_rate_hz'].flatten()
    device = data['device'].flatten()
    ai_channels = data['ai_channels'].flatten()
    ppr = data['ppr'].flatten()
    notes = data['notes'].flatten()
    created_utc = data['created_utc'].flatten()
    bias_pre = data['bias_pre']
    bias_pre_time = data['bias_pre_time'].flatten()
    bias_post = data['bias_post']
    bias_post_time = data['bias_post_time'].flatten()
    force_N = data['force_N']
    force_time = data['force_time'].flatten()
    manometers = data['manometers']
    manometers_time = data['manometers_time'].flatten()
    temperature = data['temperature'].flatten()
    pressure = data['pressure'].flatten()
    humidity = data['humidity'].flatten()
    rho = data['rho'].flatten()
    U_inf = data['U_inf'].flatten()
    D_inch = data['D_inch'].flatten()
    D_m = data['D_m'].flatten()
    pitch = data['pitch'].flatten()
    RPM = data['RPM'].flatten()
    turbulence_case = data['turbulence_case'].flatten()
    phase_angle = data['phase_angle'].flatten()
    
    if RPM == 1500:
        continue

    force_net = force_N.T - np.mean(bias_pre, axis = 1)
    Fx, Fy, Fz, Mx, My, Mz = force_net[:, 0], force_net[:, 1], force_net[:, 2], force_net[:, 3], force_net[:, 4], force_net[:, 5]
    
    time = time - time[0]
    
    omega = RPM / 60 * 2 * np.pi  # rad/s
    R_m = D_m[0] / 2
    prop_area = np.pi * R_m**2
    tip_speed = (D_m / 2) * omega  # m/s
    mu = U_inf / tip_speed
    
    mean_thrust = np.mean(-Fz[int(10*ai_rate_hz):])
    mean_torque = np.mean(-Mz[int(10*ai_rate_hz):])
    mean_power = mean_torque * omega
    
    U_inf = U_inf[0]
    RPM = RPM[0]
    
    CT = mean_thrust / (rho * prop_area * tip_speed**2)
    CQ = mean_torque / (rho * prop_area * tip_speed**2 * R_m)
    CP = mean_power / (rho * prop_area * tip_speed**3)
    
    mu_list.append(mu[0])
    CT_list.append(CT[0])
    CQ_list.append(CQ[0])
    CP_list.append(CP[0])
    RPM_values.append(RPM)
    if RPM == 3500:
        print(mean_thrust)
        print(U_inf)

# Now plot
import matplotlib.cm as cm

fig, axs = plt.subplots(1, 3, figsize=(18, 6))

sc1 = axs[0].scatter(mu_list, CT_list, c=RPM_values, cmap='viridis')
axs[0].set_xlabel('Advance Ratio (μ)')
axs[0].set_ylabel('Thrust Coefficient (CT)')
axs[0].set_title('CT vs μ')
plt.colorbar(sc1, ax=axs[0], label='RPM')

sc2 = axs[1].scatter(mu_list, CP_list, c=RPM_values, cmap='viridis')
axs[1].set_xlabel('Advance Ratio (μ)')
axs[1].set_ylabel('Power Coefficient (CP)')
axs[1].set_title('CP vs μ')
plt.colorbar(sc2, ax=axs[1], label='RPM')

sc3 = axs[2].scatter(mu_list, np.array(CT_list) / np.array(CP_list), c=RPM_values, cmap='viridis')
axs[2].set_xlabel('Advance Ratio (μ)')
axs[2].set_ylabel('Who Knows? (CT/CP)')
axs[2].set_title('CT/CP vs μ')
plt.colorbar(sc3, ax=axs[2], label='RPM')

plt.tight_layout()
plt.show()

C:\Users\Lab7\AppData\Local\Temp\ipykernel_544\4260775603.py:56: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mean_thrust = np.mean(-Fz[int(10*ai_rate_hz):])
C:\Users\Lab7\AppData\Local\Temp\ipykernel_544\4260775603.py:57: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mean_torque = np.mean(-Mz[int(10*ai_rate_hz):])


11.11964952199354
6.0
